# Task 1 — Territorial Digital Divide: Geospatial Raster Analysis

**Region:** Cusco, Peru  
**Datasets:** NASA Black Marble VNL 2025 × OSIPTEL Mobile Coverage 2019  
**Goal:** Measure the territorial digital divide by cross-referencing nighttime lights and mobile connectivity.

---
## Step 0 — Environment Setup

Import required libraries and print versions to confirm the environment is reproducible.

In [ ]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm
import scipy
from scipy import stats
from scipy.ndimage import gaussian_filter
import seaborn as sns
import pandas as pd
import rasterio
from rasterio.crs import CRS
from rasterio.warp import calculate_default_transform, reproject, Resampling

print('Library versions:')
print(f'  rasterio:   {rasterio.__version__}')
print(f'  numpy:      {np.__version__}')
print(f'  matplotlib: {matplotlib.__version__}')
print(f'  scipy:      {scipy.__version__}')
print(f'  seaborn:    {sns.__version__}')
print(f'  pandas:     {pd.__version__}')

os.makedirs('../output', exist_ok=True)
print('\nOutput directory ready.')

---
## Step 1 — Raster Loading and Inspection

Load both rasters and print CRS, shape, band count, NoData value, data type, bounding box, pixel resolution, valid pixel count, and value range.

In [ ]:
VNL_PATH  = '../data/VNL_cusco_2025.tif'
CONN_PATH = '../data/kernel_cobmovil2019_50m.tif'

def inspect_raster(path, label):
    with rasterio.open(path) as src:
        data    = src.read(1).astype(np.float64)
        nodata  = src.nodata
        res_deg = src.res                          # (row_res, col_res) in CRS units

        # Valid pixel mask
        if nodata is not None:
            valid_mask = (data != nodata) & ~np.isnan(data)
        else:
            valid_mask = ~np.isnan(data)
        valid_data = data[valid_mask]

        # Approximate km per degree (rough, valid for Cusco latitude ~-13°)
        lat_km = 111.0   # 1° latitude  ≈ 111 km
        lon_km = 111.0 * np.cos(np.radians(13))  # at -13° lat

        print(f"\n{'='*52}")
        print(f"  Raster : {label}")
        print(f"  CRS    : {src.crs}")
        print(f"  Shape  : {src.height} rows × {src.width} cols")
        print(f"  Bands  : {src.count}")
        print(f"  NoData : {nodata}")
        print(f"  Dtype  : {src.dtypes[0]}")
        print(f"  Bounds : {src.bounds}")

        if src.crs and src.crs.is_geographic:
            print(f"  Res    : {res_deg[0]:.6f}° × {res_deg[1]:.6f}°")
            print(f"  Approx : {res_deg[0]*lat_km:.3f} km × {res_deg[1]*lon_km:.3f} km")
        else:
            print(f"  Res    : {res_deg[0]:.2f} m × {res_deg[1]:.2f} m (projected)")

        print(f"  Valid  : {valid_mask.sum():,} px / {data.size:,} total")
        print(f"  Range  : [{valid_data.min():.6f}, {valid_data.max():.6f}]")

        return data, src.meta.copy(), nodata

vnl_raw,  vnl_meta,  vnl_nodata  = inspect_raster(VNL_PATH,  'NASA VNL 2025')
conn_raw, conn_meta, conn_nodata = inspect_raster(CONN_PATH, 'OSIPTEL Mobile Coverage 2019')

---
## Step 2 — Reprojection and Grid Alignment

Two-stage process:
1. Reproject connectivity from **EPSG:32719** → **EPSG:4326** (bilinear resampling).
2. Resample the reprojected layer to the **exact grid** of the VNL raster (same transform and shape).

In [ ]:
dst_crs = CRS.from_epsg(4326)

# --- Read VNL grid parameters (reference grid) ---
with rasterio.open(VNL_PATH) as vnl_src:
    vnl_transform = vnl_src.transform
    vnl_crs       = vnl_src.crs
    vnl_height    = vnl_src.height
    vnl_width     = vnl_src.width

# --- Stage 1: Reproject connectivity to EPSG:4326 ---
with rasterio.open(CONN_PATH) as conn_src:
    reproj_transform, reproj_width, reproj_height = calculate_default_transform(
        conn_src.crs, dst_crs,
        conn_src.width, conn_src.height,
        *conn_src.bounds
    )
    conn_reproj = np.zeros((reproj_height, reproj_width), dtype=np.float32)
    reproject(
        source=rasterio.band(conn_src, 1),
        destination=conn_reproj,
        src_transform=conn_src.transform,
        src_crs=conn_src.crs,
        dst_transform=reproj_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear
    )

print(f'Stage 1 — Reprojected connectivity shape: {conn_reproj.shape}  |  CRS: {dst_crs}')

# --- Stage 2: Resample to exact VNL grid ---
conn_aligned = np.zeros((vnl_height, vnl_width), dtype=np.float32)
reproject(
    source=conn_reproj,
    destination=conn_aligned,
    src_transform=reproj_transform,
    src_crs=dst_crs,
    dst_transform=vnl_transform,
    dst_crs=vnl_crs,
    resampling=Resampling.bilinear
)

print(f'Stage 2 — Aligned connectivity shape  : {conn_aligned.shape}')
print(f'          VNL shape                    : {vnl_raw.shape}')
assert conn_aligned.shape == vnl_raw.shape, 'ERROR: shape mismatch after alignment!'
print('\n✓ Both arrays share identical dimensions — grid alignment verified.')

---
## Step 3 — Robust Normalization

Apply **percentile-based normalization [2nd–98th percentile]** to both layers.  
Negative values and NoData sentinels are replaced with **0** before computing percentiles.  
Final values are clipped to **[0, 1]**.

In [ ]:
def robust_normalize(arr, nodata_val=None, p_low=2, p_high=98):
    data = arr.astype(np.float64).copy()

    # Build nodata mask before any replacement
    if nodata_val is not None:
        nodata_mask = (data == nodata_val) | np.isnan(data)
    else:
        nodata_mask = np.isnan(data)

    # Replace negative values with 0
    data[data < 0] = 0

    # Temporarily mask nodata pixels for percentile computation
    data[nodata_mask] = np.nan

    # Percentiles computed on valid (non-NaN) values only
    p2  = np.nanpercentile(data, p_low)
    p98 = np.nanpercentile(data, p_high)

    # Clip to [p2, p98] and scale to [0, 1]
    data = np.clip(data, p2, p98)
    if p98 > p2:
        data = (data - p2) / (p98 - p2)
    else:
        data = np.zeros_like(data)

    # Set nodata pixels to 0 in the final result
    data[nodata_mask] = 0.0

    return data.astype(np.float32)


vnl_norm  = robust_normalize(vnl_raw,  nodata_val=vnl_nodata)
conn_norm = robust_normalize(conn_aligned)

print('Post-normalization statistics:')
print(f'  VNL  — min: {vnl_norm.min():.4f}  max: {vnl_norm.max():.4f}  '
      f'mean: {vnl_norm.mean():.4f}  std: {vnl_norm.std():.4f}')
print(f'  Conn — min: {conn_norm.min():.4f}  max: {conn_norm.max():.4f}  '
      f'mean: {conn_norm.mean():.4f}  std: {conn_norm.std():.4f}')

---
## Step 4 — Map 1: VNL Nighttime Lights

Display raw and normalized VNL rasters side by side using the `inferno` colormap on a dark background.

In [ ]:
# Geographic extent for imshow: [west, east, south, north]
with rasterio.open(VNL_PATH) as src:
    b = src.bounds
    extent = [b.left, b.right, b.bottom, b.top]

# Prepare raw VNL for display (clip negatives and mask nodata)
vnl_display = vnl_raw.copy()
vnl_display[vnl_display < 0] = 0
if vnl_nodata is not None:
    vnl_display[vnl_display == vnl_nodata] = np.nan
vmax_raw = np.nanpercentile(vnl_display, 99)  # cap at p99 for visual clarity

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.patch.set_facecolor('black')
fig.suptitle('Alumbrado Nocturno NASA VNL 2025 — Región Cusco',
             color='white', fontsize=14, fontweight='bold', y=1.01)

for ax in axes:
    ax.set_facecolor('black')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    for spine in ax.spines.values():
        spine.set_edgecolor('white')

# Left panel — Raw
im0 = axes[0].imshow(vnl_display, extent=extent, origin='upper',
                     aspect='auto', cmap='inferno', vmin=0, vmax=vmax_raw)
axes[0].set_title('Raw (nW cm⁻² sr⁻¹)', color='white')
axes[0].set_xlabel('Longitud'); axes[0].set_ylabel('Latitud')
cb0 = plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
cb0.ax.yaxis.set_tick_params(color='white')
plt.setp(cb0.ax.yaxis.get_ticklabels(), color='white')

# Right panel — Normalized
im1 = axes[1].imshow(vnl_norm, extent=extent, origin='upper',
                     aspect='auto', cmap='inferno', vmin=0, vmax=1)
axes[1].set_title('Normalizado [0–1]', color='white')
axes[1].set_xlabel('Longitud'); axes[1].set_ylabel('Latitud')
cb1 = plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
cb1.ax.yaxis.set_tick_params(color='white')
plt.setp(cb1.ax.yaxis.get_ticklabels(), color='white')

plt.tight_layout()
plt.savefig('../output/map1_vnl.png', dpi=150, bbox_inches='tight',
            facecolor='black')
plt.show()

print('Observación: La ciudad de Cusco y el corredor Urubamba-Quillabamba '
      'son claramente visibles como manchas de alta radiancia. '
      'La mayoría del territorio (Andes remotos, selva alta) permanece oscura.')

---
## Step 5 — Map 2: Digital Divide Index (IBD) and Exclusion Index (EDT)

Compute two indices and display a direct comparison of the normalized layers, followed by individual maps for IBD and EDT.

| Index | Formula | Colormap |
|---|---|---|
| **IBD** | `VNL_norm − Connectivity_norm` | `RdYlGn_r` |
| **EDT** | `(1 − VNL_norm) × (1 − Connectivity_norm)` | `Purples` |

In [ ]:
# --- Compute indices ---
IBD = (vnl_norm - conn_norm).astype(np.float32)   # range [-1, 1]
EDT = ((1 - vnl_norm) * (1 - conn_norm)).astype(np.float32)  # range [0, 1]

# --- Figure 5a: Direct comparison — normalized VNL vs normalized Connectivity ---
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle('Comparación Directa — Alumbrado vs. Conectividad (escala común)',
             fontsize=13, fontweight='bold')

# Left — VNL (black axes face, but black labels so they show on the white figure background)
axes[0].set_facecolor('black')
axes[0].set_title('A. Alumbrado NASA VNL 2025')
axes[0].set_xlabel('Longitud')
axes[0].set_ylabel('Latitud')
# tick marks white (visible against black axes border), tick labels black (visible on white figure)
axes[0].tick_params(axis='both', color='white', labelcolor='black')
for sp in axes[0].spines.values():
    sp.set_edgecolor('#888888')

im0 = axes[0].imshow(vnl_norm, extent=extent, origin='upper',
                     aspect='auto', cmap='inferno', vmin=0, vmax=1)
cb0 = plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
cb0.set_label('Normalizado [0–1]')
cb0.ax.yaxis.set_tick_params(color='black', labelcolor='black')

# Right — Connectivity (default light background, all default black text)
im1 = axes[1].imshow(conn_norm, extent=extent, origin='upper',
                     aspect='auto', cmap='YlGnBu', vmin=0, vmax=1)
axes[1].set_title('B. Cobertura Móvil 2019')
axes[1].set_xlabel('Longitud')
axes[1].set_ylabel('Latitud')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label='Normalizado [0–1]')

plt.tight_layout()
plt.savefig('../output/map2_comparacion_directa.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Figure 5b: IBD map ---
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(IBD, extent=extent, origin='upper', aspect='auto',
               cmap='RdYlGn_r', vmin=-1, vmax=1)
ax.set_title('Índice de Brecha Digital (IBD)', fontweight='bold')
ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='IBD = VNL − Conectividad')
plt.tight_layout()
plt.savefig('../output/map2_ibd.png', dpi=150, bbox_inches='tight')
plt.show()

print('IBD — Interpretación:')
print('  Tonos rojos  → zonas con más luz que conectividad (brecha digital activa).')
print('  Tonos verdes → zonas relativamente bien conectadas respecto a su luminosidad.')
print(f'  Rango: [{IBD.min():.3f}, {IBD.max():.3f}]  |  Media: {IBD.mean():.3f}')

# --- Figure 5c: EDT map ---
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(EDT, extent=extent, origin='upper', aspect='auto',
               cmap='Purples', vmin=0, vmax=1)
ax.set_title('Exclusión Digital Total (EDT)', fontweight='bold')
ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
             label='EDT = (1−VNL) × (1−Conectividad)')
plt.tight_layout()
plt.savefig('../output/map2_edt.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nEDT — Interpretación:')
print('  Valores altos (morado intenso) → zonas sin luz NI conectividad (exclusión máxima).')
print('  Valores bajos (blanco/lila)    → zonas con al menos una de las dos infraestructuras.')
print(f'  Rango: [{EDT.min():.3f}, {EDT.max():.3f}]  |  Media: {EDT.mean():.3f}')

---
## Step 6 — Map 3: Intervention Priority

Classify territory into three priority levels based on nighttime light (population proxy) combined with low connectivity.

| Priority | Condition | Rationale |
|---|---|---|
| **P3 Critical** | VNL ≥ 0.30 and Conn < 0.10 | High population density, no internet |
| **P2 High** | VNL ≥ 0.15 and Conn < 0.15 | Urban fringe, incomplete coverage |
| **P1 Medium** | VNL ≥ 0.10 and Conn < 0.25 | Peri-urban, partial coverage |

In [ ]:
# --- Build priority layer (0 = no priority) ---
priority = np.zeros_like(vnl_norm, dtype=np.uint8)
priority[(vnl_norm >= 0.10) & (conn_norm < 0.25)] = 1   # P1 Medium
priority[(vnl_norm >= 0.15) & (conn_norm < 0.15)] = 2   # P2 High  (overwrites P1)
priority[(vnl_norm >= 0.30) & (conn_norm < 0.10)] = 3   # P3 Critical (overwrites P2)

total_px = priority.size
levels = {
    'P1 Medio'   : 1,
    'P2 Alto'    : 2,
    'P3 Crítico' : 3,
}

print('Conteo de píxeles por nivel de prioridad:')
for name, val in levels.items():
    count = (priority == val).sum()
    print(f'  {name:12s}: {count:>8,} px  ({100*count/total_px:5.2f}%)')
no_pr = (priority == 0).sum()
print(f'  {"Sin prioridad":12s}: {no_pr:>8,} px  ({100*no_pr/total_px:5.2f}%)')

# --- Discrete colormap: 0=black, 1=yellow, 2=orange, 3=red ---
cmap_p = ListedColormap(['black', 'yellow', 'orange', 'red'])
bounds_p = [-0.5, 0.5, 1.5, 2.5, 3.5]
norm_p   = BoundaryNorm(bounds_p, cmap_p.N)

fig, ax = plt.subplots(figsize=(9, 8))
fig.patch.set_facecolor('black')
ax.set_facecolor('black')
ax.set_title('Prioridad de Intervención', color='white', fontweight='bold', fontsize=13)
ax.set_xlabel('Longitud', color='white')
ax.set_ylabel('Latitud', color='white')
ax.tick_params(axis='both', color='white', labelcolor='white')
for sp in ax.spines.values():
    sp.set_edgecolor('white')

ax.imshow(priority, extent=extent, origin='upper', aspect='auto',
          cmap=cmap_p, norm=norm_p)

legend_patches = [
    mpatches.Patch(color='yellow', label='P1 Medio   — VNL≥0.10, Conn<0.25'),
    mpatches.Patch(color='orange', label='P2 Alto    — VNL≥0.15, Conn<0.15'),
    mpatches.Patch(color='red',    label='P3 Crítico — VNL≥0.30, Conn<0.10'),
]
ax.legend(handles=legend_patches, loc='lower left',
          facecolor='#222222', edgecolor='white', labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('../output/map3_prioridad.png', dpi=150, bbox_inches='tight', facecolor='black')
plt.show()

---
## Step 7 — Map 4: Social Exclusion Risk

Compute the Social Exclusion Risk score, apply a Gaussian spatial filter (σ=5) to reveal regional patterns, and display raw vs. smoothed maps side by side using the `hot_r` colormap.

In [ ]:
# --- Compute Social Exclusion Risk ---
risk_raw = EDT * (1 - vnl_norm)

# Normalize to [0, 1]
r_min, r_max = risk_raw.min(), risk_raw.max()
risk_norm_map = ((risk_raw - r_min) / (r_max - r_min)).astype(np.float32)

# Gaussian spatial smoothing (sigma=5 pixels)
risk_smooth = gaussian_filter(risk_norm_map, sigma=5).astype(np.float32)

# Percentile thresholds
p75 = np.percentile(risk_norm_map, 75)
p90 = np.percentile(risk_norm_map, 90)
print(f'Risk threshold — 75th percentile: {p75:.4f}')
print(f'  → al menos el 25% de los píxeles tiene riesgo exactamente igual a {p75:.4f}')
print(f'Risk threshold — 90th percentile: {p90:.4f}')
print(f'  → al menos el 10% de los píxeles tiene riesgo exactamente igual a {p90:.4f}')

# Proportion of pixels at maximum risk
en_maximo = (risk_norm_map == 1.0).sum()
print(f'\nPíxeles con riesgo = 1.0 (exclusión máxima): {en_maximo:,} ({100*en_maximo/risk_norm_map.size:.1f}% del territorio)')

# --- Side-by-side display ---
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.patch.set_facecolor('black')
fig.suptitle('Riesgo Exclusión Social', color='white', fontsize=14, fontweight='bold')

for ax, data, title in zip(axes,
                           [risk_norm_map, risk_smooth],
                           ['Raw', 'Suavizado (Gaussian σ=5)']):
    ax.set_facecolor('black')
    ax.set_title(title, color='white')
    ax.set_xlabel('Longitud', color='white')
    ax.set_ylabel('Latitud', color='white')
    ax.tick_params(axis='both', color='white', labelcolor='white')
    for sp in ax.spines.values():
        sp.set_edgecolor('white')
    im = ax.imshow(data, extent=extent, origin='upper', aspect='auto',
                   cmap='hot_r', vmin=0, vmax=1)
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label('Riesgo [0–1]', color='white')
    cb.ax.yaxis.set_tick_params(color='white', labelcolor='white')
    plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')

plt.tight_layout()
plt.savefig('../output/map4_riesgo_exclusion.png', dpi=150, bbox_inches='tight',
            facecolor='black')
plt.show()